# Transform Constructors Data

1. Read bronze `constructors` table
2. Keep only the columns required for Analytics (drop url column)
3. Standardise column names using snake_case
4. Rename columns to make them more meaningful
5. Filter out rows where `constructor_id` is null
6. Remove duplicated values
7. Transform values of columns in Title Case
8. Write the transformted data to a silver table 

In [0]:
%run ../00-common/01.enviroment-config

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.constructors'
silver_table = f'{catalog_name}.{silver_schema}.constructors'

## Step 1 - Read bronze `constructors` table


In [0]:
constructors_df = spark.read.table(bronze_table)

In [0]:
display(constructors_df)

## Step 2 - Keep only the columns required for Analytics (drop url column)

In [0]:
constructors_dropped_df = constructors_df.drop('url')

##Step 3 and 4 - Standardise column names using snake_case


In [0]:
constructores_df_renamed = constructors_dropped_df.withColumnsRenamed(
    {
        'constructorID': 'constructor_id',
        'name': 'constructor_name'
    }
)

## Step 5 - Filter out rows where `constructor_id` is null

In [0]:
from pyspark.sql import functions as F

In [0]:
constructors_df_not_null = constructores_df_renamed.filter(F.col('constructor_id').isNotNull())
constructors_df_not_null.count()

## Step 6 - Remove duplicated values

In [0]:
constructors_df_distinct = constructors_df_not_null.dropDuplicates(['constructor_id'])
constructors_df_distinct.count()

##Step 7 - Transform values of columns in Title Case

In [0]:
constructors_df_final = constructors_df_distinct.withColumn('nationality', F.initcap('nationality'))

## Step 8 - Write the transformted data to a silver table 

In [0]:
(
    constructors_df_final.write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(silver_table)
)

In [0]:
%sql
SELECT * FROM formula1.silver.constructors